In [ ]:
try:
  import google.colab
  IN_COLAB = True
except:
  IN_COLAB = False
print(f"In colab: {IN_COLAB}")

In [ ]:
from datetime import datetime
from pathlib import Path

if IN_COLAB:
  from google.colab import drive

  ROOT_PATH = Path('/content/drive/MyDrive/')
  OUTPUT_ROOT_PATH = ROOT_PATH / f"Colab_Data/gsm-symbolic/outputs/{datetime.now().strftime('%Y%m%d_%H%M%S')}"

  drive.mount(str(ROOT_PATH.parent))
else:
  OUTPUT_ROOT_PATH = Path(f"../../../data/gsm-symbolic/outputs/{datetime.now().strftime('%Y%m%d_%H%M%S')}").resolve()

print(OUTPUT_ROOT_PATH)

In [ ]:
if IN_COLAB:
  import os

  DRIVE_PROJECT_PATH = ROOT_PATH / "Colab_Projects"
  REPO_FOLDER_NAME = 'gsm-symbolic-benchmarking'
  FULL_REPO_PATH = DRIVE_PROJECT_PATH / REPO_FOLDER_NAME

  import os
  os.chdir(FULL_REPO_PATH)
  !git pull origin main

  try:
    import gsm_benchmarker
  except ImportError:
    !pip install transformers datasets accelerate bitsandbytes numpy pandas tqdm psutil torch cloud-tpu-client anthropic openai google-genai  # other stuff probably missing too
    !pip install -e .
    os.kill(os.getpid(), 9)  # need to restart the session for import to be possible

  os.chdir("..")

In [ ]:
import numpy as np
import logging
from huggingface_hub import login, whoami
import os

from transformers.utils.logging import disable_progress_bar
disable_progress_bar()

from gsm_benchmarker.dataset_wrapper import GSMSymbolicDataset
from gsm_benchmarker.benchmark_config import BenchmarkConfig
from gsm_benchmarker.benchmark import BenchmarkRunner, APIType
from gsm_benchmarker.models_config_parser import ModelsConfig
from gsm_benchmarker.utils.logging_setup import install_colored_logger
from gsm_benchmarker.utils.seeds import set_seed


set_seed(42)


if IN_COLAB:
  from google.colab import userdata
  hf_api_token = userdata.get('HF_TOKEN')
  os.environ["GOOGLE_API_KEY"] = userdata.get('GOOGLE_API_KEY')
  os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY')
else:
  hf_api_token = os.environ['HUGGINGFACEHUB_API_TOKEN']

login(hf_api_token)

whoami()['name']


In [ ]:
for log_name in ('urllib3', 'fsspec', 'filelock', 'h5py', 'httpcore', 'httpx', 'google_genai', 'jax', 'root', 'bitsandbytes'):
    logging.getLogger(log_name).setLevel(logging.WARNING)

logger = logging.getLogger(__name__)
install_colored_logger(level=logging.DEBUG)

In [ ]:
models_config = ModelsConfig()
all_models = models_config.all_models
print(models_config.all_models)  # note: openai - insufficient quota for benchmarking

In [ ]:
open_models = models_config.open_models
print(open_models)

In [ ]:
# these are not in the paper, but interestingly give very good results (at least on a small subset of the benchmark data, main variant)
# bigger quota than for openAI models, but likely still too small for full benchmark
other_closed_models = [
    ("gemini-2.0-flash", APIType.google_genai),
    ("gemini-2.5-flash", APIType.google_genai),
]

In [ ]:
variants = GSMSymbolicDataset.Variant

br = BenchmarkRunner(
    models=open_models,
    dset_variants=[variants.GSM8K, variants.main, variants.p1, variants.p2],
    storage_path=OUTPUT_ROOT_PATH,
    config=BenchmarkConfig(trust_remote_code=True, gpu0_max_memory="14GB", cpu_max_memory="10GB")
)

In [ ]:
br.run(n_sets=2, n_per_set=2)

In [ ]:
br.summarise_results()

In [ ]:
el = br.results[GSMSymbolicDataset.Variant.main][open_models[0]].loc[0, 0]
el

In [ ]:
print(el.question)
print()
print(el.response)